In [1]:
from transformers import pipeline

# 1. 사용할 모델 지정 (약 55MB 초경량 한국어 감정분석 모델)
MODEL_NAME = "daekeun-ml/koelectra-small-v3-nsmc"

print(f"[{MODEL_NAME}] 모델 다운로드 및 파이프라인 로딩 중...")
# pipeline 생성 시 첫 실행에서는 허깅페이스에서 가중치 파일을 자동 다운로드합니다.
classifier = pipeline(
    task="text-classification",
    model=MODEL_NAME,
    device=-1  # CPU 강제 사용 (GPU가 있다면 0)
)
print("-> 모델 로딩 완료!\n")

# 2. 테스트할 샘플 문장
test_sentences = [
    "배우들 연기도 훌륭하고 스토리 전개도 정말 몰입감 넘치네요!",
    "기대하고 봤는데 너무 지루하고 결말도 허무했습니다.",
    "그냥 시간 때우기용으로는 나쁘지 않은 정도?"
]

# 3. 모델 라벨 매핑 (0: 부정, 1: 긍정)
label_map = {
    "0": "부정 (Negative)",
    "1": "긍정 (Positive)",
    "LABEL_0": "부정 (Negative)",
    "LABEL_1": "긍정 (Positive)"
}

# 4. 추론 테스트
print("=== 감정 분석 추론 테스트 ===")
for text in test_sentences:
    # classifier 실행 결과 예시: [{'label': '1', 'score': 0.985...}]
    result = classifier(text)[0]
    raw_label = result["label"]
    confidence = result["score"]
    
    sentiment = label_map.get(raw_label, raw_label)
    
    print(f"입력: {text}")
    print(f"결과: {sentiment} (신뢰도: {confidence * 100:.2f}%)\n")

d:\vscode\aiffel\2026\11\04\my-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[daekeun-ml/koelectra-small-v3-nsmc] 모델 다운로드 및 파이프라인 로딩 중...


d:\vscode\aiffel\2026\11\04\my-project\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--daekeun-ml--koelectra-small-v3-nsmc. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\vscode\aiffel\2026\11\04\my-project\.venv\Lib\site-packages\huggingface_hub\file_downlo

-> 모델 로딩 완료!

=== 감정 분석 추론 테스트 ===
입력: 배우들 연기도 훌륭하고 스토리 전개도 정말 몰입감 넘치네요!
결과: 긍정 (Positive) (신뢰도: 99.95%)

입력: 기대하고 봤는데 너무 지루하고 결말도 허무했습니다.
결과: 부정 (Negative) (신뢰도: 99.95%)

입력: 그냥 시간 때우기용으로는 나쁘지 않은 정도?
결과: 부정 (Negative) (신뢰도: 99.12%)



In [3]:
%%writefile app/auth.py
"""
Day 6에서 만든 인증 모듈을 그대로 재사용합니다.
"""
from fastapi import HTTPException, Header

VALID_API_KEYS = {
    "test-key-001": "사용자A",
    "test-key-002": "사용자B",
}


async def verify_api_key(x_api_key: str = Header(None)) -> str:
    if x_api_key is None:
        raise HTTPException(
            status_code=401,
            detail="API Key가 필요합니다. X-API-Key 헤더를 포함해 주세요.",
        )
    if x_api_key not in VALID_API_KEYS:
        raise HTTPException(
            status_code=401,
            detail="유효하지 않은 API Key입니다.",
        )
    return VALID_API_KEYS[x_api_key]

Writing app/auth.py


In [4]:
%%writefile app/schemas.py
from pydantic import BaseModel, Field

class PredictRequest(BaseModel):
    # 1자 이상 5000자 이하의 텍스트만 허용
    text: str = Field(
        ...,
        min_length=1,
        max_length=5000,
        description="분석할 한국어 문장",
        examples=["배우들 연기도 훌륭하고 스토리 전개도 정말 몰입감 넘치네요!"]
    )

class PredictResponse(BaseModel):
    success: bool = Field(True, description="성공 여부")
    label: str = Field(..., description="예측 감정 (긍정 / 부정)")
    confidence: float = Field(..., description="예측 신뢰도 점수 (0.0 ~ 1.0)")
    raw_label: str = Field(..., description="모델의 원본 출력 라벨 ('0' 또는 '1')")

Writing app/schemas.py


In [5]:
%%writefile app/model_service.py
from transformers import pipeline

MODEL_NAME = "daekeun-ml/koelectra-small-v3-nsmc"

# 모델의 출력 라벨을 한글로 매핑 (0: 부정, 1: 긍정)
LABEL_MAP = {
    "0": "부정 (Negative)",
    "1": "긍정 (Positive)",
    "LABEL_0": "부정 (Negative)",
    "LABEL_1": "긍정 (Positive)"
}

def load_model():
    """
    Hugging Face pipeline을 이용해 감정 분석 모델을 로드하여 반환합니다.
    """
    classifier = pipeline(
        task="text-classification",
        model=MODEL_NAME,
        device=-1  # CPU 강제 사용
    )
    return classifier

def predict(model, text: str) -> dict:
    """
    텍스트 입력을 받아 감정 분석 추론 결과를 dict 형태로 반환합니다.
    """
    result = model(text)[0]
    raw_label = str(result["label"])
    confidence = float(result["score"])
    
    label = LABEL_MAP.get(raw_label, raw_label)
    
    return {
        "success": True,
        "label": label,
        "confidence": round(confidence, 4),
        "raw_label": raw_label
    }

Writing app/model_service.py


In [6]:
%%writefile app/main.py
import asyncio
from concurrent.futures import ThreadPoolExecutor
from fastapi import FastAPI, Depends, HTTPException, status
from app.auth import verify_api_key
from app.schemas import PredictRequest, PredictResponse
from app.model_service import load_model, predict

# 1. FastAPI 앱 인스턴스 생성
app = FastAPI(
    title="한국어 감정 분석 API 서비스",
    description="KoELECTRA 모델 기반 텍스트 감정 분석 서빙",
    version="1.0.0"
)

# 전역 모델 객체 및 CPU 추론용 스레드 풀
model = None
executor = ThreadPoolExecutor(max_workers=2)

# 2. startup 이벤트에서 모델 로드 (서버 기동 시 1회만 실행)
@app.on_event("startup")
async def startup_event():
    global model
    print("🚀 [FastAPI] 감정 분석 모델을 로드 중입니다...")
    model = load_model()
    print("✅ [FastAPI] 모델 로드 완료!")

# 3. GET /health 엔드포인트
@app.get("/health", tags=["System"])
async def health_check():
    return {
        "status": "healthy",
        "model_loaded": model is not None
    }

# 4. POST /predict 엔드포인트
@app.post("/predict", response_model=PredictResponse, tags=["Inference"])
async def run_predict(
    request: PredictRequest,
    user: str = Depends(verify_api_key)  # Day 6 인증 적용
):
    if model is None:
        raise HTTPException(
            status_code=status.HTTP_503_SERVICE_UNAVAILABLE,
            detail="모델이 아직 로드되지 않았습니다."
        )
    
    # CPU 연산으로 인한 비동기 이벤트 루프 블로킹을 방지하기 위해 run_in_executor 사용
    loop = asyncio.get_event_loop()
    result = await loop.run_in_executor(executor, predict, model, request.text)
    
    return result

Writing app/main.py


In [7]:
%%writefile frontend/app.py
import streamlit as st
import requests

st.set_page_config(
    page_title="한국어 감정 분석 서비스",
    page_icon="🎬",
    layout="centered"
)

st.title("🎬 영화/리뷰 감정 분석 서비스")
st.write("문장을 입력하면 AI 모델이 **긍정**인지 **부정**인지 분석해 드립니다.")

# 사이드바 설정 (API Key 및 서버 URL)
with st.sidebar:
    st.header("⚙️ API 접속 설정")
    api_url = st.text_input("백엔드 주소", value="http://localhost:8000")
    api_key = st.text_input("X-API-Key", value="test-key-001", type="password")
    
    # 헬스체크 버튼
    if st.button("서버 상태 확인"):
        try:
            res = requests.get(f"{api_url}/health", timeout=3)
            if res.status_code == 200:
                st.success("✅ 서버가 정상 동작 중입니다.")
            else:
                st.warning(f"⚠️ 상태 코드: {res.status_code}")
        except Exception as e:
            st.error(f"❌ 접속 실패: {e}")

# 메인 입력창
user_input = st.text_area(
    "분석할 리뷰/문장을 입력하세요:",
    placeholder="예시: 배우들 연기가 정말 좋았고 연출도 최고였습니다!",
    height=120
)

if st.button("감정 분석 시작", type="primary", use_container_width=True):
    if not user_input.strip():
        st.warning("분석할 텍스트를 입력해 주세요.")
    else:
        headers = {
            "Content-Type": "application/json",
            "X-API-Key": api_key
        }
        payload = {"text": user_input}
        
        with st.spinner("AI가 분석 중입니다..."):
            try:
                response = requests.post(
                    f"{api_url}/predict",
                    json=payload,
                    headers=headers,
                    timeout=5
                )
                
                if response.status_code == 200:
                    data = response.json()
                    label = data["label"]
                    confidence = data["confidence"]
                    
                    st.divider()
                    st.subheader("📊 분석 결과")
                    
                    # 긍정/부정에 따른 시각화
                    if "긍정" in label:
                        st.success(f"### 🎉 결과: {label}")
                    else:
                        st.error(f"### 💧 결과: {label}")
                    
                    st.write(f"**신뢰도 점수:** `{confidence * 100:.2f}%`")
                    st.progress(confidence)
                    
                elif response.status_code == 401:
                    st.error("🔒 인증 실패: 유효하지 않은 API Key입니다.")
                else:
                    st.error(f"서버 오류 ({response.status_code}): {response.text}")
                    
            except requests.exceptions.ConnectionError:
                st.error("🚨 FastAPI 서버에 연결할 수 없습니다. 8000번 포트로 서버가 켜져 있는지 확인하세요.")
            except Exception as e:
                st.error(f"오류 발생: {e}")

Writing frontend/app.py


---

In [11]:
%%writefile app/schemas.py
import re
from pydantic import BaseModel, Field, field_validator

class PredictRequest(BaseModel):
    text: str = Field(
        ...,
        min_length=2,
        max_length=5000,
        description="분석할 한국어 문장 (최소 2자 이상)",
        examples=["배우들 연기도 훌륭하고 스토리 전개도 정말 몰입감 넘치네요!"]
    )

    @field_validator("text")
    @classmethod
    def validate_meaningful_text(cls, v: str) -> str:
        # 1. 앞뒤 공백 제거
        cleaned = v.strip()
        
        # 2. 최소 2글자 이상인지 체크
        if len(cleaned) < 2:
            raise ValueError("문장의 길이가 너무 짧습니다. 공백을 제외하고 2자 이상 입력해 주세요.")
        
        # 3. 단순 자음/모음('ㅇ', 'ㅋㅋ')이나 특수문자만 있는지 체크 (완성형 한글 '가-힣' 또는 영문 포함 필수)
        if not re.search(r"[가-힣a-zA-Z]", cleaned):
            raise ValueError("단순 자음/모음이나 기호만으로는 분석할 수 없습니다. 완성된 단어를 포함해 주세요.")
            
        return cleaned

class PredictResponse(BaseModel):
    success: bool = Field(True, description="성공 여부")
    user: str = Field(..., description="요청한 사용자 이름")
    label: str = Field(..., description="예측 감정 (긍정 / 부정)")
    confidence: float = Field(..., description="예측 신뢰도 점수 (0.0 ~ 1.0)")
    raw_label: str = Field(..., description="모델 원본 라벨 ('0' 또는 '1')")

Overwriting app/schemas.py


In [9]:
%%writefile app/main.py
import asyncio
from concurrent.futures import ThreadPoolExecutor
from fastapi import FastAPI, Depends, HTTPException, status
from app.auth import verify_api_key
from app.schemas import PredictRequest, PredictResponse
from app.model_service import load_model, predict

app = FastAPI(
    title="한국어 감정 분석 API 서비스",
    description="KoELECTRA 모델 기반 감정 분석 서빙 API",
    version="1.0.0"
)

model = None
executor = ThreadPoolExecutor(max_workers=2)

@app.on_event("startup")
async def startup_event():
    global model
    print("🚀 [FastAPI] 감정 분석 모델을 로드 중입니다...")
    try:
        model = load_model()
        print("✅ [FastAPI] 모델 로드 완료!")
    except Exception as e:
        print(f"❌ [FastAPI] 모델 로드 실패: {e}")
        model = None

@app.get("/health", tags=["System"])
async def health_check():
    return {
        "status": "healthy" if model is not None else "degraded",
        "model_loaded": model is not None
    }

@app.post("/predict", response_model=PredictResponse, tags=["Inference"])
async def run_predict(
    request: PredictRequest,
    user: str = Depends(verify_api_key)  # 인증 실패 시 auth.py에서 401 반환
):
    # 1. 모델이 준비되지 않은 경우 (503 Service Unavailable)
    if model is None:
        raise HTTPException(
            status_code=status.HTTP_503_SERVICE_UNAVAILABLE,
            detail="모델이 아직 준비되지 않았거나 로드에 실패했습니다."
        )
    
    # 2. 비동기 스레드 풀에서 추론 수행 및 모델 예외 처리 (500 Internal Server Error)
    try:
        loop = asyncio.get_running_loop()
        result = await loop.run_in_executor(executor, predict, model, request.text)
        result["user"] = user  # 인증된 사용자 정보 포함
        return result
    except Exception as e:
        raise HTTPException(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
            detail=f"추론 처리 중 모델 오류가 발생했습니다: {str(e)}"
        )

Overwriting app/main.py


In [10]:
%%writefile frontend/app.py
import streamlit as st
import requests

st.set_page_config(
    page_title="한국어 감정 분석 서비스",
    page_icon="🎬",
    layout="centered"
)

st.title("🎬 영화/리뷰 감정 분석 서비스")
st.write("문장을 입력하면 AI 모델이 **긍정**인지 **부정**인지 분석해 드립니다.")

# 사이드바 설정
with st.sidebar:
    st.header("⚙️ API 접속 설정")
    api_url = st.text_input("백엔드 주소", value="http://localhost:8000")
    api_key = st.text_input("X-API-Key", value="test-key-001", type="password")
    
    if st.button("서버 상태 확인"):
        try:
            res = requests.get(f"{api_url}/health", timeout=3)
            if res.status_code == 200:
                data = res.json()
                if data.get("model_loaded"):
                    st.success("✅ 서버 및 모델 정상 준비 완료")
                else:
                    st.warning("⚠️ 서버는 동작 중이나 모델 로드 실패")
            else:
                st.warning(f"⚠️ 상태 코드: {res.status_code}")
        except Exception as e:
            st.error(f"❌ 접속 실패: {e}")

# 메인 텍스트 입력창
user_input = st.text_area(
    "분석할 리뷰/문장을 입력하세요:",
    placeholder="예시: 배우들 연기가 정말 좋았고 연출도 최고였습니다!",
    height=120
)

if st.button("감정 분석 시작", type="primary", use_container_width=True):
    # 클라이언트 단에서의 1차 공백 체크
    if not user_input.strip():
        st.warning("⚠️ 공백을 제외한 문장을 최소 1자 이상 입력해 주세요.")
    else:
        headers = {
            "Content-Type": "application/json",
            "X-API-Key": api_key
        }
        payload = {"text": user_input}
        
        with st.spinner("AI가 분석 중입니다..."):
            try:
                response = requests.post(
                    f"{api_url}/predict",
                    json=payload,
                    headers=headers,
                    timeout=5
                )
                
                # 1. 성공 (200 OK)
                if response.status_code == 200:
                    data = response.json()
                    label = data["label"]
                    confidence = data["confidence"]
                    user_name = data.get("user", "사용자")
                    
                    st.divider()
                    st.caption(f"🔑 인증 확인: **{user_name}** 님으로 분석되었습니다.")
                    st.subheader("📊 분석 결과")
                    
                    if "긍정" in label:
                        st.success(f"### 🎉 결과: {label}")
                    else:
                        st.error(f"### 💧 결과: {label}")
                    
                    st.write(f"**신뢰도 점수:** `{confidence * 100:.2f}%`")
                    st.progress(confidence)
                
                # 2. 인증 실패 (401 Unauthorized)
                elif response.status_code == 401:
                    st.error("🔒 인증 실패(401): 유효하지 않은 API Key입니다. 사이드바 설정을 확인하세요.")
                
                # 3. 입력값 검증 실패 (422 Unprocessable Entity)
                elif response.status_code == 422:
                    error_detail = response.json().get("detail", "입력 데이터 형식이 잘못되었습니다.")
                    st.warning(f"⚠️ 입력 검증 실패(422): {error_detail}")
                
                # 4. 모델 준비 안 됨 (503 Service Unavailable)
                elif response.status_code == 503:
                    st.error("⏳ 서버는 켜져 있으나 모델이 아직 로드되지 않았습니다(503). 잠시 후 다시 시도해 주세요.")
                
                # 5. 모델 추론 에러 (500 Internal Server Error)
                elif response.status_code == 500:
                    error_msg = response.json().get("detail", "서버 내부 오류")
                    st.error(f"🚨 모델 추론 중 오류 발생(500): {error_msg}")
                    
                else:
                    st.error(f"기타 오류 발생 (코드: {response.status_code}): {response.text}")
                    
            except requests.exceptions.ConnectionError:
                st.error("🚨 FastAPI 서버에 연결할 수 없습니다. 터미널에서 서버(8000번)가 켜져 있는지 확인하세요.")
            except Exception as e:
                st.error(f"통신 에러 발생: {e}")

Overwriting frontend/app.py
